# 第 17 讲：综合模拟赛 I：读题、建模与代码

> 用一个综合小题串起数据处理、评价模型和优化模型，产出初版结果。

## 课件导学

**任务情境**：综合模拟赛要求把读题、评价、优化和图表输出串成一条可复现链路。

**关键概念**

- 综合评价
- 预算约束
- 站点选择
- 结果表
- 初版报告

**本讲产物**

- site_scores.csv
- selected_sites.csv
- mini_contest_scores.png

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [COMAP MCM/ICM 赛事页](https://www.comap.com/contests/mcm-icm)：理解开放式应用题的综合建模特点。
- [SciPy linprog 文档](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html)：对照优化模型的标准表述。
- [pandas groupby 用户指南](https://pandas.pydata.org/docs/user_guide/groupby.html)：复用表格汇总和分组分析技能。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-17"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
cities = pd.DataFrame({
    "city": [f"C{i}" for i in range(1, 9)],
    "demand": rng.integers(80, 180, 8),
    "cost": rng.integers(40, 90, 8),
    "accessibility": rng.uniform(0.5, 0.95, 8),
    "risk": rng.uniform(0.05, 0.30, 8),
})
directions = {"demand": "max", "cost": "min", "accessibility": "max", "risk": "min"}
def normalize(df, directions):
    out = pd.DataFrame(index=df.index)
    for col, direction in directions.items():
        if direction == "max":
            out[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
        else:
            out[col] = (df[col].max() - df[col]) / (df[col].max() - df[col].min())
    return out
norm = normalize(cities.set_index("city"), directions)
weights = pd.Series({"demand": 0.35, "cost": 0.25, "accessibility": 0.25, "risk": 0.15})
cities["score"] = (norm * weights).sum(axis=1).values
cities.sort_values("score", ascending=False).to_csv(OUTPUT_ROOT / "site_scores.csv", index=False, encoding="utf-8-sig")
cities.sort_values("score", ascending=False)

In [ ]:
budget = 220
best = None
for bits in itertools.product([0, 1], repeat=len(cities)):
    chosen = np.array(bits)
    total_cost = (chosen * cities["cost"]).sum()
    if total_cost <= budget and chosen.sum() > 0:
        total_score = (chosen * cities["score"]).sum()
        if best is None or total_score > best["total_score"]:
            best = {"selection": bits, "total_cost": total_cost, "total_score": total_score}
selected = cities.loc[np.array(best["selection"]) == 1, ["city", "cost", "score"]]
selected.to_csv(OUTPUT_ROOT / "selected_sites.csv", index=False, encoding="utf-8-sig")
best, selected

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cities.sort_values("score").plot.barh(x="city", y="score", ax=ax, legend=False)
ax.set_title("Mini contest site evaluation")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "mini_contest_scores.png", dpi=160)
plt.show()

## 实战环节

**课堂任务**

- 读懂站点选择任务的输入输出。
- 完成评价分数和预算选择。
- 输出一张排名图和一张最终选择表。

**课后挑战**：把权重改成三种策略，比较最终选择是否稳定。

**验收清单**

- 是否形成完整数据到结果链
- 是否保存中间表
- 是否能把结果交给论文手使用

In [ ]:
lesson_resources = [{'title': 'COMAP MCM/ICM 赛事页', 'url': 'https://www.comap.com/contests/mcm-icm', 'reading_note': '理解开放式应用题的综合建模特点。'}, {'title': 'SciPy linprog 文档', 'url': 'https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html', 'reading_note': '对照优化模型的标准表述。'}, {'title': 'pandas groupby 用户指南', 'url': 'https://pandas.pydata.org/docs/user_guide/groupby.html', 'reading_note': '复用表格汇总和分组分析技能。'}]
lesson_deliverables = ['site_scores.csv', 'selected_sites.csv', 'mini_contest_scores.png']
lesson_checklist = ['是否形成完整数据到结果链', '是否保存中间表', '是否能把结果交给论文手使用']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")